In [2]:
import pandas as pd
import numpy as np
import glob
import os

from xgboost import XGBRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt




In [4]:
data_folder = "data/raw"
csv_files = sorted(glob.glob(os.path.join(data_folder, "*.csv")))
print("Found files:", csv_files)

all_dfs = []
for file in csv_files:
    city_name = os.path.basename(file).split("_weather")[0]
    df_city = pd.read_csv(file, skiprows=3)
    df_city.columns = ['date', 'temp_max', 'temp_min', 'precip', 'wind_max', 'humidity', 'pressure', 'weathercode']
    df_city['city'] = city_name
    all_dfs.append(df_city)

df = pd.concat(all_dfs, ignore_index=True)

# Clean
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['city', 'date']).reset_index(drop=True)

print("Shape:", df.shape)
print("Nulls:", df.isnull().sum().sum())
df.head()


Found files: ['data/raw/Chennai_weather.csv', 'data/raw/Delhi_weather.csv', 'data/raw/Guwahati_weather.csv', 'data/raw/Jaipur_weather.csv', 'data/raw/Kolkata_weather.csv', 'data/raw/Mumbai_weather.csv', 'data/raw/Pune_weather.csv', 'data/raw/Shimla_weather.csv']
Shape: (29232, 9)
Nulls: 0


,date,temp_max,temp_min,precip,wind_max,humidity,pressure,weathercode,city
0,2015-01-01,28.7,22.0,0.2,10.2,87,1009.7,51,Chennai
1,2015-01-02,28.6,23.3,0.8,13.2,87,1012.2,51,Chennai
2,2015-01-03,28.4,23.0,2.0,9.7,90,1012.8,51,Chennai
3,2015-01-04,28.3,21.8,0.5,10.7,85,1012.9,51,Chennai
4,2015-01-05,28.4,21.6,0.1,14.4,83,1012.6,51,Chennai


In [5]:
# Step 3: season
df['month'] = df['date'].dt.month

# Step 4: city as a number
df['city_code'] = df['city'].astype('category').cat.codes

# Check the mapping
print(df[['city', 'city_code']].drop_duplicates().to_string(index=False))
df.head()


    city  city_code
 Chennai          0
   Delhi          1
Guwahati          2
  Jaipur          3
 Kolkata          4
  Mumbai          5
    Pune          6
  Shimla          7


,date,temp_max,temp_min,precip,wind_max,humidity,pressure,weathercode,city,month,city_code
0,2015-01-01,28.7,22.0,0.2,10.2,87,1009.7,51,Chennai,1,0
1,2015-01-02,28.6,23.3,0.8,13.2,87,1012.2,51,Chennai,1,0
2,2015-01-03,28.4,23.0,2.0,9.7,90,1012.8,51,Chennai,1,0
3,2015-01-04,28.3,21.8,0.5,10.7,85,1012.9,51,Chennai,1,0
4,2015-01-05,28.4,21.6,0.1,14.4,83,1012.6,51,Chennai,1,0


In [6]:
city_list = []

for city in df['city'].unique():
    city_df = df[df['city'] == city].copy()    # only this city's rows (already sorted by date)

    # Step 5: past values (shift(1) = yesterday, shift(2) = 2 days ago ...)
    city_df['temp_lag1']     = city_df['temp_max'].shift(1)
    city_df['temp_lag2']     = city_df['temp_max'].shift(2)
    city_df['temp_lag3']     = city_df['temp_max'].shift(3)
    city_df['temp_lag7']     = city_df['temp_max'].shift(7)
    city_df['humidity_lag1'] = city_df['humidity'].shift(1)
    city_df['pressure_lag1'] = city_df['pressure'].shift(1)

    # Step 6: average temp of the last 7 days
    city_df['temp_mean7'] = city_df['temp_max'].rolling(7).mean()

    city_list.append(city_df)

df = pd.concat(city_list, ignore_index=True)

df[['date', 'city', 'temp_max', 'temp_lag1', 'temp_lag2', 'temp_lag7', 'temp_mean7']].head(10)


,date,city,temp_max,temp_lag1,temp_lag2,temp_lag7,temp_mean7
0,2015-01-01,Chennai,28.7,NaN,NaN,NaN,NaN
1,2015-01-02,Chennai,28.6,28.7,NaN,NaN,NaN
2,2015-01-03,Chennai,28.4,28.6,28.7,NaN,NaN
3,2015-01-04,Chennai,28.3,28.4,28.6,NaN,NaN
4,2015-01-05,Chennai,28.4,28.3,28.4,NaN,NaN
5,2015-01-06,Chennai,27.7,28.4,28.3,NaN,NaN
6,2015-01-07,Chennai,28.0,27.7,28.4,NaN,28.300000
7,2015-01-08,Chennai,27.6,28.0,27.7,28.7,28.142857
8,2015-01-09,Chennai,26.8,27.6,28.0,28.6,27.885714
9,2015-01-10,Chennai,27.0,26.8,27.6,28.4,27.685714


In [7]:
city_list = []

for city in df['city'].unique():
    city_df = df[df['city'] == city].copy()

    # Step 7: future temp_max (shift(-1) = tomorrow, shift(-2) = day after ...)
    for day in range(1, 8):
        city_df[f'target_d{day}'] = city_df['temp_max'].shift(-day)

    city_list.append(city_df)

df = pd.concat(city_list, ignore_index=True)

target_cols = [f'target_d{day}' for day in range(1, 8)]
print(target_cols)
df[['date', 'city', 'temp_max', 'target_d1', 'target_d2', 'target_d7']].head(10)


['target_d1', 'target_d2', 'target_d3', 'target_d4', 'target_d5', 'target_d6', 'target_d7']


,date,city,temp_max,target_d1,target_d2,target_d7
0,2015-01-01,Chennai,28.7,28.6,28.4,27.6
1,2015-01-02,Chennai,28.6,28.4,28.3,26.8
2,2015-01-03,Chennai,28.4,28.3,28.4,27.0
3,2015-01-04,Chennai,28.3,28.4,27.7,26.4
4,2015-01-05,Chennai,28.4,27.7,28.0,26.7
5,2015-01-06,Chennai,27.7,28.0,27.6,27.2
6,2015-01-07,Chennai,28.0,27.6,26.8,26.7
7,2015-01-08,Chennai,27.6,26.8,27.0,27.7
8,2015-01-09,Chennai,26.8,27.0,26.4,27.6
9,2015-01-10,Chennai,27.0,26.4,26.7,28.1


In [8]:
print("Before:", len(df))
df = df.dropna().reset_index(drop=True)
print("After: ", len(df))
print("Nulls left:", df.isnull().sum().sum())


Before: 29232
After:  29120
Nulls left: 0


In [9]:
feature_cols = [
    # today's values
    'temp_max', 'temp_min', 'humidity', 'pressure', 'precip', 'wind_max',
    # past values
    'temp_lag1', 'temp_lag2', 'temp_lag3', 'temp_lag7',
    'humidity_lag1', 'pressure_lag1',
    # average, season, city
    'temp_mean7', 'month', 'city_code'
]
print(len(feature_cols), "features")


15 features


In [10]:
train_end = pd.Timestamp('2022-01-01')   # same as LSTM notebook
val_end   = pd.Timestamp('2023-07-03')   # same as LSTM notebook

# Leave a 7-day gap before each cut, so a row's future targets never fall in the next set
gap = pd.Timedelta(days=7)

train = df[df['date'] < train_end - gap]
val   = df[(df['date'] >= train_end) & (df['date'] < val_end - gap)]
test  = df[df['date'] >= val_end]

print("Train:", train['date'].min().date(), "to", train['date'].max().date(), len(train))
print("Val:  ", val['date'].min().date(),   "to", val['date'].max().date(),   len(val))
print("Test: ", test['date'].min().date(),  "to", test['date'].max().date(),  len(test))


Train: 2015-01-08 to 2021-12-24 20344
Val:   2022-01-01 to 2023-06-25 4328
Test:  2023-07-03 to 2024-12-25 4336


In [11]:
X_train = train[feature_cols]
X_val   = val[feature_cols]
X_test  = test[feature_cols]

y_train = train[target_cols]
y_val   = val[target_cols]
y_test  = test[target_cols]

print("X_train:", X_train.shape, " y_train:", y_train.shape)
print("X_val:  ", X_val.shape,   " y_val:  ", y_val.shape)
print("X_test: ", X_test.shape,  " y_test: ", y_test.shape)


X_train: (20344, 15)  y_train: (20344, 7)
X_val:   (4328, 15)  y_val:   (4328, 7)
X_test:  (4336, 15)  y_test:  (4336, 7)


In [12]:
model_d1 = XGBRegressor(
    n_estimators=500,          # max number of trees
    learning_rate=0.05,        # how much each tree corrects
    max_depth=6,               # how complex each tree can be
    early_stopping_rounds=50   # stop if val doesn't improve for 50 trees
)

model_d1.fit(
    X_train, y_train['target_d1'],
    eval_set=[(X_val, y_val['target_d1'])],
    verbose=50                 # print progress every 50 trees
)

print("Best number of trees:", model_d1.best_iteration)

# Check on test
pred_d1     = model_d1.predict(X_test)
baseline_d1 = X_test['temp_max']         # persistence: tomorrow = today

print(f"Baseline day-1 RMSE: {root_mean_squared_error(y_test['target_d1'], baseline_d1):.2f} °C")
print(f"XGBoost  day-1 RMSE: {root_mean_squared_error(y_test['target_d1'], pred_d1):.2f} °C")


[0]	validation_0-rmse:6.33409
[50]	validation_0-rmse:1.47912
[100]	validation_0-rmse:1.35948
[150]	validation_0-rmse:1.35658
[200]	validation_0-rmse:1.35708
[237]	validation_0-rmse:1.35684
Best number of trees: 187
Baseline day-1 RMSE: 1.34 °C
XGBoost  day-1 RMSE: 1.26 °C


In [14]:
models = {}

for day in range(1, 8):
    target = f'target_d{day}'

    model = XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        early_stopping_rounds=50
    )
    model.fit(
        X_train, y_train[target],
        eval_set=[(X_val, y_val[target])],
        verbose=False              # no printing inside the loop
    )

    models[day] = model
    print(f"Day {day} trained, best trees: {model.best_iteration}")


Day 1 trained, best trees: 187
Day 2 trained, best trees: 125
Day 3 trained, best trees: 253
Day 4 trained, best trees: 281
Day 5 trained, best trees: 208
Day 6 trained, best trees: 201
Day 7 trained, best trees: 196


In [21]:
xgb_rmse = []
baseline_rmse = []

print("Day | Baseline | XGBoost | MAE  | within ±2°C | Better?")
for day in range(1, 8):
    actual   = y_test[f'target_d{day}']
    pred     = models[day].predict(X_test)
    baseline = X_test['temp_max']          # persistence: same as today for every day

    rmse_b = root_mean_squared_error(actual, baseline)
    rmse_x = root_mean_squared_error(actual, pred)
    mae    = mean_absolute_error(actual, pred)
    r2     = r2_score(actual, pred)
    acc2   = (np.abs(actual - pred) <= 2).mean() * 100

    baseline_rmse.append(rmse_b)
    xgb_rmse.append(rmse_x)

    better = "✅" if rmse_x < rmse_b else "❌"
    print(f" {day}  |  {rmse_b:.2f}    |  {rmse_x:.2f}   | {mae:.2f} |  |   {acc2:.1f}%    | {better}")

print(f"\nAverage → Baseline: {np.mean(baseline_rmse):.2f} °C | XGBoost: {np.mean(xgb_rmse):.2f} °C")


Day | Baseline | XGBoost | MAE  | within ±2°C | Better?
 1  |  1.34    |  1.26   | 0.91 |  |   89.8%    | ✅
 2  |  1.77    |  1.59   | 1.18 |  |   83.0%    | ✅
 3  |  2.03    |  1.79   | 1.33 |  |   78.8%    | ✅
 4  |  2.20    |  1.89   | 1.41 |  |   75.9%    | ✅
 5  |  2.33    |  1.92   | 1.45 |  |   74.5%    | ✅
 6  |  2.43    |  1.97   | 1.49 |  |   73.5%    | ✅
 7  |  2.52    |  2.02   | 1.53 |  |   72.5%    | ✅

Average → Baseline: 2.09 °C | XGBoost: 1.78 °C


In [16]:
lstm_rmse = [1.30, 1.64, 1.82, 1.94, 2.02, 2.08, 2.14]

print("Day | Baseline | LSTM | XGBoost | Better ?")
for day in range(7):
    winner = "XGBoost" if xgb_rmse[day] < lstm_rmse[day] else "LSTM"
    print(f" {day+1}  |  {baseline_rmse[day]:.2f}    | {lstm_rmse[day]:.2f} |  {xgb_rmse[day]:.2f}   | {winner}")


Day | Baseline | LSTM | XGBoost | Better ?
 1  |  1.34    | 1.30 |  1.26   | XGBoost
 2  |  1.77    | 1.64 |  1.59   | XGBoost
 3  |  2.03    | 1.82 |  1.79   | XGBoost
 4  |  2.20    | 1.94 |  1.89   | XGBoost
 5  |  2.33    | 2.02 |  1.92   | XGBoost
 6  |  2.43    | 2.08 |  1.97   | XGBoost
 7  |  2.52    | 2.14 |  2.02   | XGBoost


In [23]:
def show_forecast(city, start):
    today = pd.Timestamp(start) - pd.Timedelta(days=1)          # the "today" row before the forecast
    row = test[(test['city'] == city) & (test['date'] == today)]

    X_row = row[feature_cols]
    table = pd.DataFrame({
        'city':          city,
        'forecast_date': pd.date_range(start, periods=7).date,
        'actual':        row[target_cols].values[0],
        'predicted':     [models[day].predict(X_row)[0] for day in range(1, 8)],
        'baseline':      row['temp_max'].values[0],
    }).round(1)
    table['error'] = (table['predicted'] - table['actual']).round(1)
    return table

pd.concat([
    show_forecast('Pune',   '2024-06-01'),
    show_forecast('Delhi',  '2024-05-15'),
    show_forecast('Shimla', '2024-12-10'),
], ignore_index=True)


,city,forecast_date,actual,predicted,baseline,error
0,Pune,2024-06-01,35.9,35.400002,35.8,-0.5
1,Pune,2024-06-02,36.2,34.799999,35.8,-1.4
2,Pune,2024-06-03,37.4,33.900002,35.8,-3.5
3,Pune,2024-06-04,31.4,34.000000,35.8,2.6
4,Pune,2024-06-05,33.2,33.799999,35.8,0.6
5,Pune,2024-06-06,33.6,33.200001,35.8,-0.4
6,Pune,2024-06-07,33.9,33.200001,35.8,-0.7
7,Delhi,2024-05-15,42.3,41.000000,40.7,-1.3
8,Delhi,2024-05-16,42.7,40.000000,40.7,-2.7
9,Delhi,2024-05-17,44.0,40.299999,40.7,-3.7
